# 4. Hosted model: Gemini vs Gemma 3 4B

The public demo answers with Google's Gemini API instead of running Gemma in the Space (the free
Space gets no GPU). This notebook evaluates Gemini on the same 50 hand-labelled questions and the
same setups as notebook 3, so the two models can be compared directly.

**Before running:** add your API key as a Colab secret named `GEMINI_API_KEY` (🔑 icon in the left
sidebar, enable notebook access). **Runtime:** a T4 GPU speeds up the embeddings; CPU also works. Requests are paced to stay within the
free-tier rate limit; the run takes about 10 minutes.

Note: on the free tier, Google may use prompts to improve its products. The prompts here contain
only public law text and the public test questions.

In [ ]:
!pip -q install sentence-transformers sacrebleu google-genai pandas pyarrow
!rm -rf amharic-legal-rag && git clone -q https://github.com/4brhamy/amharic-legal-rag.git
import sys, json, time, os
sys.path.insert(0, "amharic-legal-rag")
import numpy as np, pandas as pd
from google.colab import userdata
from amlegal.retrieval import HybridRetriever
from amlegal.generate import build_messages, cited_articles, is_refusal
from amlegal.llm import gemini_generator, DEFAULT_GEMINI_MODEL

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
MODEL = "auto-lite"  # Flash-Lite: its free daily quota fits the 200-call evaluation
generate = gemini_generator(model=MODEL)
print(MODEL, "->", generate([{"role": "user", "content": "ሰላም! በአንድ ቃል መልስ፦ የኢትዮጵያ ዋና ከተማ ማን ናት?"}], 20))

corpus = [json.loads(l) for l in open("amharic-legal-rag/data/family_code.jsonl", encoding="utf-8")]
by_article = {r["article"]: r for r in corpus}
qa = pd.read_parquet("hf://datasets/beimnet777/Family-code-of-Ethiopia-QA-TEST/data/train-00000-of-00001.parquet")
qa.columns = ["question", "answer"]
gold = json.load(open("amharic-legal-rag/data/test_labels.json"))
qa["gold"] = [gold[str(i)] for i in range(len(qa))]

## Retrieval (same hybrid retriever as the demo; BGE-M3 takes ~1 min on a GPU, ~10 on CPU)

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
embedder = SentenceTransformer("BAAI/bge-m3", device="cuda" if torch.cuda.is_available() else "cpu")
retriever = HybridRetriever(corpus, encode=lambda t: embedder.encode(t, normalize_embeddings=True, batch_size=16))
retrieved = [retriever.search(q, k=5) for q in qa.question]
print("gold article in top 5:", sum(bool({a["article"] for a in r} & set(g)) for r, g in zip(retrieved, qa.gold)), "/ 50")

## Answer the 50 questions in four setups

In [ ]:
CLOSED_BOOK = ("አንተ የኢትዮጵያን የቤተሰብ ሕግ የምታውቅ ረዳት ነህ። ጥያቄውን በአማርኛ በአጭሩ መልስ፤ "
               "የምታውቀውን አንቀጽ በ [አንቀጽ ቁጥር] መልክ ጥቀስ።")
SETUPS = {
    "closed_book": lambda q, i: [{"role": "system", "content": CLOSED_BOOK}, {"role": "user", "content": q}],
    "rag_k5_en_strict": lambda q, i: build_messages(q, retrieved[i]),
    "rag_k3_am_lenient": lambda q, i: build_messages(q, retrieved[i][:3], include_english=False, lenient=True),
    "oracle": lambda q, i: build_messages(q, [by_article[a] for a in qa.gold[i]]),
}
from concurrent.futures import ThreadPoolExecutor, as_completed
WORKERS = 3  # answers take a few seconds each, so 3 in flight stays under the free per-minute limit


def answer(name, i):
    try:
        ans = generate(SETUPS[name](qa.question[i], i))
        return ans, generate.last_model
    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"[:300], None


rows, t0 = [], time.time()
todo = [(name, i) for name in SETUPS for i in range(len(qa))]
with ThreadPoolExecutor(WORKERS) as pool:
    futures = {pool.submit(answer, name, i): (name, i) for name, i in todo}
    for n, fut in enumerate(as_completed(futures), 1):
        name, i = futures[fut]
        ans, used = fut.result()
        rows.append({"i": i, "setup": name, "answer": ans, "model": used})
        if n % 25 == 0:
            print(f"{n}/{len(todo)} done, {time.time() - t0:.0f}s")
order = {name: k for k, name in enumerate(SETUPS)}
rows.sort(key=lambda r: (order[r["setup"]], r["i"]))
res = pd.DataFrame(rows)
print("errors:", res.answer.str.startswith("ERROR").sum(), "| models used:", res.model.value_counts().to_dict())

## Score (same metrics as notebook 3)

In [ ]:
from sacrebleu.metrics import CHRF
chrf = CHRF()
res["gold"] = res.i.map(qa.gold)
res["chrF"] = [chrf.sentence_score(a, [qa.answer[i]]).score for a, i in zip(res.answer, res.i)]
res["cited"] = res.answer.map(cited_articles)
res["refusal"] = res.answer.map(is_refusal)
res["citation_correct"] = [bool(c) and c[0] in g for c, g in zip(res.cited, res.gold)]
res["answered"] = ~res.refusal
summary = res.groupby("setup").agg(
    chrF=("chrF", "mean"), answers=("answered", "mean"),
    right_overall=("citation_correct", lambda s: (s & res.loc[s.index, "answered"]).mean()),
    refusals=("refusal", "mean"))
summary["right_when_answering"] = summary.right_overall / summary.answers
summary["wrong_overall"] = summary.answers - summary.right_overall
summary = summary.loc[list(SETUPS)].round(3)
summary

In [ ]:
for i in [7, 37, 42]:
    print(f"\n=== Q{i}: {qa.question[i]}\ngold {qa.gold[i]}")
    for s in SETUPS:
        print(f"--- {s}: {res[(res.i == i) & (res.setup == s)].answer.iloc[0][:300]}")

In [ ]:
os.makedirs("results", exist_ok=True)
summary.to_csv(f"results/gemini_summary.csv")
res[["i", "setup", "model", "answer", "cited", "chrF", "citation_correct", "refusal"]].to_json(
    "results/gemini_answers.jsonl", orient="records", lines=True, force_ascii=False)
with open("results/gemini_model.txt", "w") as f:
    f.write(MODEL + "\n")
!zip -qr gemini_results.zip results && ls -la gemini_results.zip
from google.colab import files
files.download("gemini_results.zip")